
Your system should:
1. **Classify** the incoming question to determine what type of information is needed
2. **Select** the appropriate bash tool(s) to retrieve relevant context
3. **Execute** the bash commands and collect the output
4. **Pass** the retrieved context along with the question to an LLM
5. **Generate** an answer with references to specific files

**Bash Tools to Consider:**
- `grep` / `rg` (ripgrep) - Search file contents for patterns
- `find` - Locate files by name or pattern
- `cat` / `head` / `tail` - Read file contents
- `tree` - Visualize directory structure
- `ls` - List directory contents

**Test Questions:**

Your system should be able to answer the following questions:

| # | Question | Difficulty |
|---|----------|------------|
| 1 | "What Python dependencies does this project use?" | Simple |
| 2 | "What is the main entry point file for the registry service?" | Simple |
| 3 | "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)" | Simple |
| 4 | "How does the authentication flow work, from token validation to user authorization?" | Difficult |
| 5 | "What are all the API endpoints available in the registry service and what scopes do they require?" | Difficult |
| 6 | "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?" | Very Hard |

*Note: Questions 4-6 require synthesizing information from multiple files (code + docs + configs).*

**Hints:**
- Different question types need different retrieval strategies
- Structure questions might use `tree` or `find`
- Code search questions might use `grep` with file type filters
- Dependency questions should look at `pyproject.toml` and `package.json`
- Documentation questions should search the `docs/` folder

In [4]:
import os
import json
import logging
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")

from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

[2026-03-06 01:44:54,864] p9999 {1115267388.py:17} INFO - GROQ_API_KEY is set


/home/ubuntu/dsan6725/a3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_9999/1115267388.py:24: LangChainDeprecationWarning: The class `ChatLiteLLM` was deprecated in LangChain 0.3.24 and will be removed in 1.0. An updated version of the class exists in the `langchain-litellm package and should be used instead. To use it run `pip install -U `langchain-litellm` and import as `from `langchain_litellm import ChatLiteLLM``.
  llm = ChatLiteLLM(
[2026-03-06 01:45:11,638] p9999 {1115267388.py:28} INFO - Using model: groq/llama-3.1-8b-instant


In [3]:
"""
QUERY ROUTING
"""
from sre_parse import State
import time
from langgraph.types import Command

def classify_query(query: str) -> str:
    # Structure questions might use tree or find
    # Code search questions might use grep with file type filters
    # Dependency questions should look at pyproject.toml and package.json
    # Documentation questions should search the docs/ folder

    # use LLM to classify the query  
    system_prompt = """
        You are a query classification tool for a codebase retrieval system.

        Classify the user query into exactly ONE of the following four categories('code search', 'file structure', 'documentation', 'dependencies') based on the main intent of the question:
        
        1. code search:
        Questions about how code works, authentication flows, API endpoints, functions, classes, logic, or implementation details.
        
        2. file structure:
        Questions about the organization of the repository, directory layout, or what files and languages exist in the project.
        
        3. documentation: 
        Questions asking for explanations, guides, entry points, or descriptions likely found in documentation files (docs/, README, markdown).
        
        4. dependencies:
        Questions about libraries, packages, requirements, or external dependencies used by the project.

        Rules:
        - Choose exactly ONE category('code search', 'file structure', 'documentation', 'dependencies').
        - Output ONLY the category name.
        - Do not explain your answer.
        - Do not include any extra text.

        Classify the following query:
    """

    wait = 1
    for i in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\nQuestion: {query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2
    
    return response.content

/tmp/ipykernel_9999/1889266844.py:4: DeprecationWarning: module 'sre_parse' is deprecated
  from sre_parse import State


In [16]:
"""
BASH TOOL RETRIEVAL
Chunking is not necessary because we do not use vector search. 
Different question types need different retrieval strategies
Structure questions might use tree or find
Code search questions might use grep with file type filters
Dependency questions should look at pyproject.toml and package.json
Documentation questions should search the docs/ folder
"""
import subprocess
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
import re

stop_words = set(stopwords.words("english"))

def extract_keywords(query):
    # words = re.findall(r"\w+", query.lower())
    # return [w for w in words if w not in stop_words and len(w) > 3]
    system_prompt = """
    You are a retrieval augmented generation AI tool that takes in a query and provides a list of words, phrases, or lemmas that can be searched for in the codebase using 'grep' to find relevant context. 
    
    Rules:
    Only output the list of words, phrases, or lemmas separated by commas. Do not include any other text. 
    Only include FIVE total outputs, separated by commas.
    Use strings that are likely to be found in a codebase.
     
    Perform this task for the following query:
    """

    wait = 1
    for _ in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(f"Prompt: {system_prompt}\n{query.lower()}")
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")
        wait *= 2
    
    query_words = response.content
    query_words = [w.strip() for w in query_words.split(",")]
    return query_words

def run_bash(cmd: str, max_output: int) -> str:
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )

    output = result.stdout
    return output[:max_output]
    # return output

# run_bash("cat ../mcp-gateway-registry/pyproject.toml ../mcp-gateway-registry/package.json", max_output=6000)

def retrieve_context(query: str, q_class: str) -> str:
    max_output = 15000
    file_names = []
    cmds = []
    if q_class == "file structure":
        cmd = "tree -L 5 ../mcp-gateway-registry"
        cmds.append(cmd)
        cmd = "find ../mcp-gateway-registry -type f"
        cmds.append(cmd)
        cmd = "cat ../mcp-gateway-registry/README.md ../mcp-gateway-registry/pyproject.toml ../mcp-gateway-registry/package.json"
        cmds.append(cmd)
        context = run_bash(cmd, max_output=max_output)

    elif q_class == "dependencies":
        cmd = "cat ../mcp-gateway-registry/pyproject.toml ../mcp-gateway-registry/package.json "
        file_names.extend(["pyproject.toml", "package.json"])
        cmds.append(cmd)
        context = run_bash(cmd, max_output=max_output)

    elif q_class == "documentation":
        context = ""
        query_words = extract_keywords(query)
        for word in query_words:
            cmd = f"grep -R -n -C 10 '{word}' ../mcp-gateway-registry/docs"
            cmds.append(cmd)
            context += run_bash(cmd, max_output=max_output//len(query_words))
            file_names.extend(run_bash(cmd + ' -l', max_output=max_output).splitlines())

    elif q_class == "code search":
        context = ""
        query_words = extract_keywords(query)
        print(query_words)
        for word in query_words:
            # cmd = f"grep -R -n -C 10 '{word}' ../mcp-gateway-registry"
            cmd = f"grep -R -C 10 '{word}' --include='*.py' ../mcp-gateway-registry"
            #     --include='*.ts' \
            #     --include='*.js' \
            #     --include='*.yaml' \
            #     --include='*.yml' \
            #     --include='*.json' \
                # '{word}' ../mcp-gateway-registry"
            cmds.append(cmd)
            context += run_bash(cmd, max_output=max_output//len(query_words))
            file_names.extend(run_bash(cmd + ' -l', max_output=max_output).splitlines())
    else:
        context = ""
        query_words = extract_keywords(query)
        for word in query_words:
            cmd = f"grep -R -C 10 '{word}' ../mcp-gateway-registry"
            cmds.append(cmd)
            context += run_bash(cmd, max_output=max_output//len(query_words))
            file_names.extend(run_bash(cmd + ' -l', max_output=max_output).splitlines())
    logger.info(f"\n\nBash command: {cmd}\n")
    logger.info(f"Retrieved context length: {len(context)}\n")
    return [context, cmds, file_names]


[nltk_data] Downloading package stopwords to /home/ubuntu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
import time

system_prompt = (
    "You are an assistant for question-answering tasks regarding the mcp-gateway-registry codebase. "
    "Use the following pieces of retrieved context from the codebase to answer "
    "the question. Cite your sources by filename."
    "\n\n"
)

questions = ["What Python dependencies does this project use?", "What is the main entry point file for the registry service?", "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)", "How does the authentication flow work, from token validation to user authorization?", "What are all the API endpoints available in the registry service and what scopes do they require?", "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?"]

output_file = "part1_results.txt"
open(output_file, "w").close()

for q in questions:
    wait = 1
    classification = classify_query(q)
    logger.info(f"Classified query as: {classification}")

    context, cmd, file_names = retrieve_context(q, classification)
    # print("\n\n\n")
    print(len(context))
    # context = context[:10000]
    # print("\n\n\n")

    for i in range(5):
        time.sleep(wait)
        try:
            response = llm.invoke(
                f"Prompt: {system_prompt}\nContext:\n{context}\n\nQuestion:{q}"
            )
            ans = response.content
            break
        except Exception as e:
            logger.error(f"Error invoking LLM: {e}")

        wait *= 2  

    with open(output_file, "a") as f:
        f.write(f"Question: {q}\n")
        f.write(f"Classification: {classification}\n")  
        f.write(f"Commands: {cmd}\n")
        # f.write(f"Context:\n{contexts[i]}\n")
        # f.write(f"Files: {', '.join(file_names)}\n")
        f.write(f"Answer: {ans}\n\n")

01:54:35 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:54:35,641] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
01:54:35 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 01:54:35,801] p9999 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 01:54:35,804] p9999 {2453509391.py:18} INFO - Classified query as: dependencies
[2026-03-06 01:54:35,810] p9999 {633316799.py:112} INFO - 

Bash command: cat ../mcp-gateway-registry/pyproject.toml ../mcp-gateway-registry/package.json 

[2026-03-06 01:54:35,810] p9999 {633316799.py:113} INFO - Retrieved context length: 5126



5126


01:54:36 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:54:36,813] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
01:54:37 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 01:54:37,943] p9999 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
01:54:38 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:54:38,947] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
01:54:39 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 01:54:39,106] p9999 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-06 01:54:39,108] p9999 {2453509391.py:18} INFO - Classified query as: code search
01:54:40 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion(

['main', 'entry', 'point', 'registry', 'service', 'main.py']
15000


01:54:41 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:54:41,459] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:54:41,523] p9999 {2453509391.py:35} ERROR - Error invoking LLM: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kex0x720fceb9hdmwxymt5e8` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4034, Requested 3618. Please try again in 16.52s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



...

01:56:48 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:56:48,511] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:56:48,600] p9999 {2453509391.py:35} ERROR - Error invoking LLM: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kex0x720fceb9hdmwxymt5e8` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5114, Requested 2890. Please try again in 20.04s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



01:56:56 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:56:56,602] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:56:56,672] p9999 {2453509391.py:35} ERROR - Error invoking LLM: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kex0x720fceb9hdmwxymt5e8` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4305, Requested 2890. Please try again in 11.95s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}




Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



01:57:12 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-06 01:57:12,674] p9999 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
01:57:14 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-06 01:57:14,141] p9999 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
